# Forget-MI LoKU — Kaggle Notebook

> 🎯 **Mục đích**: Chạy LoKU trên Kaggle để đo song song với baseline `run_kaggle_baseline.ipynb` trên cùng GPU T4 — đảm bảo so sánh FAIR cho thesis Chương 4.

## Cấu trúc

| Cell | Mục đích |
|---|---|
| 1 | Setup Kaggle env: clone repo, install deps, restore CSV |
| 2 | Auto-detect Kaggle dataset paths + verify |
| 3 | Define helpers (run_multiseed, aggregate_summary) |
| 4a/4b/4c | Train multi-seed 3% / 6% / 10% (4b/4c có RUN flag, mặc định OFF) |
| 5 | Bảng tổng hợp cross-forget% |
| 6 | Push results lên GitHub |

## Setup Kaggle 1 lần

1. **Datasets** (Sidebar → + Add data):
   - `forget-mi-data` (chứa `data/metadata` + `data/img_data`)
   - `forget-mi-models` (chứa `base_model/` + `retrained_model/`)
2. **Secrets** (Add-ons → Secrets): `GITHUB_TOKEN`, `GIT_EMAIL`, `GIT_NAME`
3. **GPU**: Settings → Accelerator → GPU T4 x2

## So với baseline

| | Baseline (run_kaggle_baseline.ipynb) | LoKU (notebook này) |
|---|---|---|
| Script | `forgetmi_partial.py` | `forgetmi_loku.py` |
| Config | `config_baseline_kaggle.yaml` | `config_loku_kaggle.yaml` |
| Trainable | 113M (full FT) | ~3M (LoRA + FILA, ~2.6%) |
| Epochs | 30 | 8 |
| Time/seed | ~3h | ~15-20 phút |
| Checkpoint | ~450 MB | ~12 MB |
| CSV path | `/kaggle/working/results_summary.csv` | `/kaggle/working/unlearning_output/results_summary.csv` |
| Repo CSV | `results_summary_kaggle.csv` | `results_summary_loku_kaggle.csv` |


In [ ]:
# ====================================
# CELL 1: Setup Kaggle env (clone repo + install deps + restore CSV)
# ====================================
import os, sys, subprocess, shutil

WORK_DIR = "/kaggle/working"
REPO_URL = "https://github.com/nhnhu146/Forget-MI-LoKU.git"
REPO_NAME = "Forget-MI-LoKU"
REPO_DIR = f"{WORK_DIR}/{REPO_NAME}"

os.chdir(WORK_DIR)

# 1. Clone hoặc pull repo
if not os.path.exists(REPO_DIR):
    print(f"🔽 Clone {REPO_URL}")
    subprocess.run(['git', 'clone', REPO_URL], check=True)
else:
    print(f"🔄 Pull latest")
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/master'], check=True)

os.chdir(REPO_DIR)
print(f"📂 CWD: {os.getcwd()}")
subprocess.run(['git', 'log', '--oneline', '-1'])

# 2. ⭐ RESTORE CSV từ repo về /kaggle/working/unlearning_output/ (nếu Cell 6 trước đó đã push)
csv_in_repo = "experiments/results_summary_loku_kaggle.csv"
csv_target = "/kaggle/working/unlearning_output/results_summary.csv"
os.makedirs(os.path.dirname(csv_target), exist_ok=True)
if os.path.exists(csv_in_repo) and not os.path.exists(csv_target):
    shutil.copy(csv_in_repo, csv_target)
    n = sum(1 for _ in open(csv_target)) - 1
    print(f"♻️  Restored CSV từ repo: {csv_target} ({n} rows)")
elif os.path.exists(csv_target):
    n = sum(1 for _ in open(csv_target)) - 1
    print(f"📊 CSV hiện có: {csv_target} ({n} rows)")
else:
    print(f"📊 CSV chưa có — chạy từ đầu")

# 3. Install deps
print("\n📦 Installing deps...")
# FORCE-INSTALL pinned versions (Kaggle pre-installs newer transformers
# which breaks ImageTextModel due to BertAttention API changes in 4.45+)
print("📦 Force-install pinned deps (transformers 4.38.0 + peft 0.10.0)...")
get_ipython().system('pip install -q pydicom scikit-image pyyaml')
get_ipython().system('pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"')
print("✅ Pinned deps installed.")

# 4. Check GPU
import torch
if torch.cuda.is_available():
    print(f"\n🟢 GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("\n🔴 KHÔNG CÓ GPU — LoKU sẽ KHÔNG xong trong 12h Kaggle limit!")
    print("   → Settings → Accelerator → GPU T4 x2 → Save → Restart Session")

print("\n✅ Setup complete")


In [ ]:
# ====================================
# CELL 2: Auto-detect Kaggle dataset paths
# ====================================
import os, glob


def _find_kaggle_dataset(slug):
    direct = f'/kaggle/input/{slug}'
    if os.path.isdir(direct):
        return direct
    candidates = glob.glob(f'/kaggle/input/datasets/*/{slug}')
    return candidates[0] if candidates else None


print("🔍 Auto-detect Kaggle dataset paths:\n")

mimic_data_root = _find_kaggle_dataset('forget-mi-data')
mimic_models_root = (_find_kaggle_dataset('forget-mi-models-full')
                     or _find_kaggle_dataset('forget-mi-models-v2')
                     or _find_kaggle_dataset('forget-mi-models'))

print(f"📦 forget-mi-data   → {mimic_data_root or '❌ NOT FOUND'}")
print(f"📦 forget-mi-models → {mimic_models_root or '❌ NOT FOUND'}\n")

KAGGLE_MIMIC_DATA_ROOT = mimic_data_root
KAGGLE_MIMIC_MODELS_ROOT = mimic_models_root

# Dựng base_model + gold theo nesting thực tế (full vs cũ)
MIMIC_BASE_MODEL = None
MIMIC_GOLD = {3: None, 6: None, 10: None}
if mimic_models_root:
    if 'forget-mi-models-full' in mimic_models_root:
        MIMIC_BASE_MODEL = f"{mimic_models_root}/original_model/forgetme/training_original_model"
        MIMIC_GOLD = {
            3:  f"{mimic_models_root}/model_retrained_3per/model_retrained_3per",
            6:  f"{mimic_models_root}/model_retrained_6per/model_retrained_6per",
            10: f"{mimic_models_root}/model_retrained_10per/model_retrained_10per",
        }
    else:
        MIMIC_BASE_MODEL = f"{mimic_models_root}/base_model/training_original_model"
        MIMIC_GOLD = {3: f"{mimic_models_root}/retrained_model/model_retrained_3per", 6: None, 10: None}
    MIMIC_GOLD = {k: (v if v and os.path.isdir(v) else None) for k, v in MIMIC_GOLD.items()}

if mimic_data_root and mimic_models_root:
    checks = {
        "Base model":      f"{MIMIC_BASE_MODEL}/pytorch_model.bin",
        "Retrained 3%":    (f"{MIMIC_GOLD[3]}/pytorch_model.bin" if MIMIC_GOLD.get(3) else "— (N/A)"),
        "Text metadata":   f"{mimic_data_root}/data/metadata",
        "Image data":      f"{mimic_data_root}/data/img_data",
        "MIMIC split CSV": "./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv",
        "Synonyms":        "./data_splits/Synonyms.csv",
        "Forget 3% list":  "./data_splits/forget_set_3per.csv",
    }
    all_ok = True
    for name, path in checks.items():
        exists = os.path.exists(path)
        print(f"  {'✅' if exists else '❌'} {name:<18} {path}")
        if not exists and name != "Retrained 3%":
            all_ok = False
    print()
    if all_ok:
        print("✅ Tất cả paths OK — sẵn sàng chạy LoKU")
    else:
        print("⚠️  Thiếu paths — kiểm tra Kaggle Datasets đã add đúng chưa")
else:
    print("❌ Thiếu Kaggle Dataset. Thêm vào notebook Sidebar → + Add data:")
    print("   - forget-mi-data")
    print("   - forget-mi-models-full")


In [ ]:
# ====================================
# CELL 3: Helpers cho LoKU multi-seed (BẮT BUỘC trước Cell 4*)
# ====================================
import os, numpy as np, pandas as pd
from datetime import datetime

CSV_PATH = "/kaggle/working/unlearning_output/results_summary.csv"

PAPER_REF = {
    3:  {"MIA_paper": 0.571, "Df_AUC": 0.735, "Df_F1": 0.393, "Dt_AUC": 0.625, "Dt_F1": 0.250, "Time_h": 5.0},
    6:  {"MIA_paper": 0.615, "Df_AUC": 0.654, "Df_F1": 0.328, "Dt_AUC": 0.599, "Dt_F1": 0.270, "Time_h": 5.0},
    10: {"MIA_paper": 0.810, "Df_AUC": 0.656, "Df_F1": 0.313, "Dt_AUC": 0.565, "Dt_F1": 0.252, "Time_h": 5.0},
}


def _gold_path(forget_pct):
    # Gold từ MIMIC_GOLD (Cell 2) — hỗ trợ 3/6/10%% với forget-mi-models-full.
    return globals().get('MIMIC_GOLD', {}).get(forget_pct)


METRIC_DEFS = [
    ('MIA',                 'MIA_persample',   None,         '↓'),
    ('MIA_paper',           'MIA_paper',       'MIA_paper',  '↓'),
    ('forget_ce',           'forget_ce',       None,         '·'),
    ('test_ce',             'test_ce',         None,         '·'),
    ('Df_AUC',              'Forget AUC',      'Df_AUC',     '↓'),
    ('Df_F1',               'Forget Mac-F1',   'Df_F1',      '↓'),
    ('Dt_AUC',              'Test AUC',        'Dt_AUC',     '↑'),
    ('Dt_F1',               'Test Mac-F1',     'Dt_F1',      '↑'),
    ('dist_vs_re',          '1 − CosSim',      None,         '↓'),
    ('unlearn_time_hours',  'Time (h)',        'Time_h',     '↓'),
    ('gpu_peak_GB',         'GPU peak (GB)',   None,         '·'),
    ('trainable_ratio',     'Trainable ratio', None,         '↓'),
]


def _seed_done(forget_pct, seed):
    if not os.path.exists(CSV_PATH):
        return False
    try:
        df = pd.read_csv(CSV_PATH)
    except Exception:
        return False
    if 'forget_pct' not in df.columns or 'seed' not in df.columns:
        return False
    mask = df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per") & (df['seed'] == seed)
    return bool(mask.any())


def run_multiseed(forget_pct, seeds=(42, 123, 7), ihl=1.25, force_redo=False):
    if isinstance(seeds, int):
        seeds = (seeds,)
    seeds = tuple(seeds)
    assert forget_pct in (3, 6, 10)
    forget_csv = f"./data_splits/forget_set_{forget_pct}per.csv"
    gold_path = _gold_path(forget_pct)
    has_gold = gold_path is not None
    paper = PAPER_REF[forget_pct]
    exp_name = f"final_ihl{int(ihl*100):03d}_{forget_pct}per"

    print(f"\n{'#'*72}")
    print(f"# 🎯 LoKU — FORGET {forget_pct}% — seeds={list(seeds)} — IHL={ihl}")
    print(f"# Forget CSV    : {forget_csv}")
    if has_gold:
        print(f"# Gold retrained: ✅ {gold_path}")
    else:
        print(f"# Gold retrained: ❌ N/A (1−CosSim KHÔNG hợp lệ)")
    print(f"{'#'*72}\n")

    if not os.path.exists(forget_csv):
        raise FileNotFoundError(f"Forget set không tồn tại: {forget_csv}")

    OVR = (f"forget_set_path={forget_csv},id=loku_{forget_pct}per,ihl_forget_weight={ihl},"
           f"base_model_path={MIMIC_BASE_MODEL},"
           f"bert_pretrained_dir={MIMIC_BASE_MODEL},"
           f"text_data_dir={KAGGLE_MIMIC_DATA_ROOT}/data/metadata,"
           f"img_data_dir={KAGGLE_MIMIC_DATA_ROOT}/data/img_data")
    if has_gold:
        OVR += f",retrained_model_path={gold_path}"

    HYP = (f"LoKU Kaggle FORGET {forget_pct}%: distill_teacher=og, early_stop=val, "
           f"IHL={ihl}, image-FILA scale0.5. Multi-seed mean+-std.")

    for i, s in enumerate(seeds):
        if not force_redo and _seed_done(forget_pct, s):
            print(f"⏭️  SEED {s} đã có trong CSV — skip (force_redo=True để rerun)\n")
            continue
        print(f"\n{'='*60}\n🎲 SEED {s} ({i+1}/{len(seeds)}) ▸ LoKU FORGET {forget_pct}%\n{'='*60}")
        cmd = (f'PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py '
               f'--config config_loku_kaggle.yaml --fresh --seed {s} '
               f'--override "{OVR}" '
               f'--exp {exp_name}_seed{s} --hypothesis "{HYP}"')
        get_ipython().system(cmd)

    aggregate_summary(forget_pct, seeds, exp_name, paper, has_gold, ihl=ihl)


def aggregate_summary(forget_pct, seeds, exp_name, paper_ref, has_gold, ihl=1.25):
    if not os.path.exists(CSV_PATH):
        print(f"❌ Không tìm thấy {CSV_PATH}. Bỏ qua aggregate.")
        return
    df = pd.read_csv(CSV_PATH)
    mask = df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per") & df['seed'].isin(seeds)
    df = df[mask]
    if df.empty:
        print(f"❌ Không có rows cho {forget_pct}% + seeds={list(seeds)}.")
        return
    if 'timestamp' in df.columns:
        df = df.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')

    print(f"\n{'='*100}")
    print(f"📊 LoKU MULTI-SEED — FORGET {forget_pct}%, seeds={list(seeds)}")
    if not has_gold:
        print(f"⚠️  Gold retrained N/A cho {forget_pct}% → 1−CosSim KHÔNG hợp lệ")
    print('=' * 100)
    hdr = (f"{'Metric':<20}" + "".join(f"{f'seed{s}':>10}" for s in seeds)
           + f"{'mean ± std':>18}" + f"{'paper':>10}" + f"{'Δ vs paper':>12}")
    print(hdr); print("-" * len(hdr))

    md_rows = [
        "| Metric | " + " | ".join(f"seed {s}" for s in seeds)
        + " | **mean ± std** | Paper | Δ vs paper |",
        "|---|" + "---|" * (len(seeds) + 3),
    ]

    for csv_key, label, paper_key, direction in METRIC_DEFS:
        if csv_key not in df.columns:
            continue
        vals = []
        for s in seeds:
            sub = df[df['seed'] == s]
            if sub.empty:
                continue
            try:
                v = float(sub[csv_key].iloc[-1])
            except Exception:
                continue
            if v != v:
                continue
            vals.append(v)
        if not vals:
            continue
        m, sd = float(np.mean(vals)), float(np.std(vals))
        inv = " ⚠️" if (csv_key == 'dist_vs_re' and not has_gold) else ""

        if paper_key and paper_key in paper_ref:
            p_val = paper_ref[paper_key]
            delta = m - p_val
            arrow_good = (direction == '↓' and delta < 0) or (direction == '↑' and delta > 0)
            sym = "✅" if arrow_good else ("❌" if direction in ('↓', '↑') else "·")
            paper_str = f"{p_val:>10.3f}"
            delta_str = f"{sym}{delta:+.3f}"
        else:
            paper_str = f"{'—':>10}"
            delta_str = "—"

        cell_vals = "".join(f"{v:>10.3f}" for v in vals) + " " * (10 * (len(seeds) - len(vals)))
        ms_str = f"{m:>10.3f}±{sd:.3f}"
        print(f"{label+inv:<20}{cell_vals}{ms_str:>18}{paper_str}{delta_str:>12}")

        md_vals = " | ".join(f"{v:.3f}" for v in vals) + " | " * (len(seeds) - len(vals))
        md_paper = f"{paper_ref[paper_key]:.3f}" if paper_key and paper_key in paper_ref else "—"
        md_delta = f"{delta:+.3f}" if paper_key and paper_key in paper_ref else "—"
        md_rows.append(f"| {label}{inv} | {md_vals} | **{m:.3f} ± {sd:.3f}** | {md_paper} | {md_delta} |")

    os.makedirs("experiments", exist_ok=True)
    out_md = f"experiments/summary_{exp_name}_kaggle_multiseed.md"
    body_lines = [
        f"# LoKU Kaggle Multi-seed — FORGET {forget_pct}% — {exp_name}",
        "",
        f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}_",
        "",
        f"**Config**: `final_ihl{int(ihl*100):03d}` (honest, no F_re), IHL={ihl}, image-FILA scale=0.3",
        "",
        "**Hardware**: Kaggle T4 (so sánh trực tiếp được với baseline run_kaggle_baseline.ipynb)",
        "",
        f"**Seeds**: {list(seeds)}",
        "",
        "**Gold retrained**: " + ("✅ available" if has_gold else "❌ N/A — 1−CosSim KHÔNG hợp lệ"),
        "",
    ]
    body_lines.extend(md_rows)
    body_lines.extend([
        "",
        f"**Paper Forget-MI ({forget_pct}%)**: MIA={paper_ref['MIA_paper']} | Df_AUC={paper_ref['Df_AUC']} | "
        f"Dt_AUC={paper_ref['Dt_AUC']} | Time≈{paper_ref['Time_h']}h",
        "",
        "_Δ vs paper_: âm = LoKU tốt hơn (↓ metrics) hoặc kém hơn (↑ metrics). ✅ = LoKU thắng.",
    ])
    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join(body_lines))
    print(f"\n💾 Summary MD: {out_md}")


print("✅ Helpers đã định nghĩa: run_multiseed(), aggregate_summary()")
print(f"   CSV path: {CSV_PATH}")
print("   Config  : config_loku_kaggle.yaml")


---

## 🎯 Training cells — chạy theo nhu cầu

| Cell | Forget% | Có RUN flag? | Mặc định |
|---|---|---|---|
| 4a | 3% | Không | ✅ Chạy |
| 4b | 6% | `RUN_6PER` | ❌ Skip |
| 4c | 10% | `RUN_10PER` | ❌ Skip |

Save Version chạy unattended → chỉ Cell 4a active. Khi muốn thêm 6%/10% → mở cell tương ứng set `True` rồi Save Version mới.

In [ ]:
# ====================================
# CELL 4a: LoKU MULTI-SEED FORGET 3% (gold ✅)
# ====================================
# Expected (Colab reference): MIA≈0.43, Df_AUC≈0.74, Dt_AUC≈0.68, Time≈0.2h
# Trên T4 Kaggle có thể chậm hơn Colab nhẹ.

SEEDS_THIS_RUN = (42, 123, 7)
run_multiseed(forget_pct=3, seeds=SEEDS_THIS_RUN, ihl=1.25)


In [ ]:
# ====================================
# CELL 4b: LoKU MULTI-SEED FORGET 6% (no gold)
# ====================================
RUN_6PER = True    # ⚙️ Set True khi muốn train 6%
SEEDS_THIS_RUN = (42, 123, 7)

if RUN_6PER:
    run_multiseed(forget_pct=6, seeds=SEEDS_THIS_RUN, ihl=1.25)
else:
    print("⏭️  Skip LoKU 6% (RUN_6PER=False)")


In [ ]:
# ====================================
# CELL 4c: LoKU MULTI-SEED FORGET 10% (no gold)
# ====================================
RUN_10PER = True    # ⚙️ Set True khi muốn train 10%
SEEDS_THIS_RUN = (42, 123, 7)

if RUN_10PER:
    run_multiseed(forget_pct=10, seeds=SEEDS_THIS_RUN, ihl=1.25)
else:
    print("⏭️  Skip LoKU 10% (RUN_10PER=False)")


---
## 🔬 CELL 4g: ABLATION (P1) — 5 biến thể LoKU / MIMIC 3%

Component knockout (Fisher · Image-FILA · IHL) + rank (r=4, r=16). **Chạy CELL 1 trước** (repo cần cờ `loku_random_init`). ~1h tổng → điền 4 bảng ablation.

In [ ]:
# ====================================================================
# CELL 4g: ABLATION (P1) — 5 biến thể LoKU trên MIMIC 3%
# component knockout (Fisher / Image-FILA / IHL) + rank (r=4, r=16)
# ⚠️ Chạy CELL 1 trước để repo có cờ loku_random_init. ~1h tổng.
# Kết quả cuối cell → điền 4 bảng ablation trong luận văn.
# ====================================================================
import os, glob, subprocess, pandas as pd

def find1(pat):
    h = sorted(glob.glob(pat, recursive=True)); return h[0] if h else None
def model_dir(p):
    if not p: return None
    if os.path.isfile(os.path.join(p, 'pytorch_model.bin')): return p
    b = find1(os.path.join(p, '**', 'pytorch_model.bin')); return os.path.dirname(b) if b else p

og   = model_dir(find1('/kaggle/input/**/training_original_model'))
re3  = model_dir(find1('/kaggle/input/**/model_retrained_3per'))
meta = find1('/kaggle/input/**/forget-mi-data/data/metadata')
img  = find1('/kaggle/input/**/forget-mi-data/data/img_data')
for n, p in [('og', og), ('re3', re3), ('meta', meta), ('img', img)]:
    print(f"  {n:5}= {p}")
assert all([og, re3, meta, img]), "❌ Thiếu path MIMIC — kiểm tra đã Add forget-mi-data + forget-mi-models(-full)"

BASE  = ["ihl_forget_weight=1.25", "loku_image_subtract_scale=0.5", "unlearn_epochs=8",
         "kappa_cls_retain=2.0", "lora_r=8", "lora_image_last_k_blocks=1"]   # = config D
PATHS = [f"base_model_path={og}", f"bert_pretrained_dir={og}", f"retrained_model_path={re3}",
         f"text_data_dir={meta}", f"img_data_dir={img}",
         "forget_set_path=./data_splits/forget_set_3per.csv",
         "output_dir=/kaggle/working/abl_output"]

# 3 knockout thành phần TRƯỚC (quan trọng nhất), rank SAU (cắt r16 trước nếu hụt quota)
ABLATIONS = [
    ("abl_lora_random", ["loku_random_init=1"]),          # tắt Fisher/FILA  -> abl-fisher
    ("abl_text_only",   ["lora_image_last_k_blocks=0"]),  # tắt Image-FILA   -> abl-modality
    ("abl_ihl_zero",    ["ihl_forget_weight=0"]),         # tắt IHL (lambda=0) -> abl-ihl
    ("abl_rank4",       ["lora_r=4"]),                    # r=4              -> abl-rank
    ("abl_rank16",      ["lora_r=16"]),                   # r=16
]
for tag, extra in ABLATIONS:
    OVR = ",".join(PATHS + BASE + extra + [f"id={tag}"])   # extra sau BASE -> ghi đè lora_r/ihl
    print(f"\n{'='*60}\n🔬 {tag}  (+{extra})\n{'='*60}")
    subprocess.run(["python", "training/forgetmi_loku.py",
                    "--config", "config_loku_kaggle.yaml", "--override", OVR],
                   env={**os.environ, "WANDB_MODE": "disabled", "PYTHONPATH": "."}, check=True)

# ---- đọc kết quả ----
paths = set(glob.glob('/kaggle/working/**/results_summary*.csv', recursive=True))
df = pd.concat([pd.read_csv(x) for x in paths], ignore_index=True)
cols = ['id', 'MIA', 'MIA_paper', 'Df_AUC', 'Df_F1', 'Dt_AUC', 'Dt_F1', 'forget_ce', 'test_ce', 'lora_r', 'ihl']
print("\n\n========== KẾT QUẢ ABLATION ==========")
print(df[df['id'].astype(str).str.startswith('abl_')][cols].drop_duplicates('id', keep='last').to_string(index=False))


---

## 🔬 Indiana University CXR — generalization check (LoKU)

Yêu cầu: Kaggle Datasets **`forget-mi-data-iu`** + **`forget-mi-models-iu`** (tạo từ
`preprocess_iu_kaggle.ipynb` + `run_kaggle_train_iu.ipynb`). Cell dưới auto-detect paths,
ghi CSV riêng `/kaggle/working/unlearning_iu_output/results_summary.csv` (KHÔNG đụng MIMIC).
Nếu chưa add 2 dataset IU → cell in cảnh báo rồi skip (không crash).

In [ ]:
# ====================================
# CELL 4d (IU): LoKU MULTI-SEED — Indiana University CXR — FORGET 3%
# ====================================
import os, glob, numpy as np, pandas as pd, shutil
from datetime import datetime

def _find_one(_pats):
    for _p in _pats:
        _h = glob.glob(_p, recursive=True)
        if _h:
            return sorted(_h, key=len)[0]
    return None

# --- Detect IU inputs (recursive — robust với mọi nesting của Kaggle) ---
_tsv = _find_one(['/kaggle/input/**/all_data.tsv'])
IU_META = os.path.dirname(_tsv) if _tsv else None
IU_IMG = None
if IU_META:
    _sib = os.path.join(os.path.dirname(IU_META), 'img_data')
    IU_IMG = _sib if os.path.isdir(_sib) else _find_one(['/kaggle/input/**/img_data'])
# Cách 3: ảnh không đóng gói trong forget-mi-data-iu → trỏ sang raddar (attach kèm).
if not IU_IMG:
    for _r in glob.glob('/kaggle/input/*chest-xray*') + glob.glob('/kaggle/input/datasets/*/*chest-xray*'):
        for _sub in ('images/images_normalized', 'images/images', 'images_normalized', 'images'):
            _p = os.path.join(_r, _sub)
            if os.path.isdir(_p) and glob.glob(os.path.join(_p, '*.png')):
                IU_IMG = _p; break
        if IU_IMG: break
IU_SPLIT = _find_one(['/kaggle/input/**/iu-split.csv'])
IU_FORGET3 = _find_one(['/kaggle/input/**/forget_set_3per_iu.csv'])
_og = _find_one(['/kaggle/input/**/model_og_IU/pytorch_model.bin'])
IU_OG_DIR = os.path.dirname(_og) if _og else None
_re3 = _find_one(['/kaggle/input/**/model_retrained_iu_3per/pytorch_model.bin'])
IU_RE3_DIR = os.path.dirname(_re3) if _re3 else None

CSV_PATH_IU = "/kaggle/working/unlearning_iu_output/results_summary.csv"

# Restore IU CSV từ repo (skip seeds đã xong ở session trước)
_iu_repo_csv = "experiments/results_summary_loku_iu_kaggle.csv"
if os.path.exists(_iu_repo_csv) and not os.path.exists(CSV_PATH_IU):
    os.makedirs(os.path.dirname(CSV_PATH_IU), exist_ok=True)
    shutil.copy(_iu_repo_csv, CSV_PATH_IU)
    print("♻️  Restored IU CSV từ repo")


def _iu_seed_done(pct, seed):
    if not os.path.exists(CSV_PATH_IU):
        return False
    try:
        df = pd.read_csv(CSV_PATH_IU)
    except Exception:
        return False
    if 'seed' not in df.columns:
        return False
    m = (df['seed'] == seed)
    if 'forget_pct' in df.columns:
        m = m & df['forget_pct'].astype(str).str.contains(f"_{pct}per")
    return bool(m.any())


def run_loku_iu(forget_pct=3, seeds=(42, 123, 7), ihl=1.25, force_redo=False, tag='', extra_ovr=''):
    if isinstance(seeds, int):
        seeds = (seeds,)
    seeds = tuple(seeds)
    if not all([IU_META, IU_IMG, IU_SPLIT, IU_FORGET3, IU_OG_DIR]):
        print("❌ IU chưa sẵn sàng — cần forget-mi-data-iu + forget-mi-models-iu:")
        for n, v in [("metadata", IU_META), ("img_data", IU_IMG), ("split", IU_SPLIT),
                     ("forget3", IU_FORGET3), ("model_og", IU_OG_DIR), ("model_re3", IU_RE3_DIR)]:
            print(f"   {'✅' if v else '❌'} {n}: {v}")
        return
    forget_csv = IU_FORGET3 if forget_pct == 3 else _find_one([f'/kaggle/input/**/forget_set_{forget_pct}per_iu.csv'])
    has_gold = (IU_RE3_DIR is not None) and (forget_pct == 3)
    exp_name = f"loku_iu_{forget_pct}per{tag}"
    print(f"\n{'#'*72}")
    print(f"# 🔬 LoKU IU — FORGET {forget_pct}% — seeds={list(seeds)} — IHL={ihl}")
    print(f"# base_model : {IU_OG_DIR}")
    print(f"# gold       : {'✅ ' + IU_RE3_DIR if has_gold else '❌ N/A → 1−CosSim không hợp lệ'}")
    print(f"{'#'*72}")

    OVR = (f"forget_set_path={forget_csv},id={exp_name},ihl_forget_weight={ihl},"
           f"base_model_path={IU_OG_DIR},bert_pretrained_dir={IU_OG_DIR},"
           f"text_data_dir={IU_META},img_data_dir={IU_IMG},"
           f"data_split_path={IU_SPLIT},output_channel_encoding=multiclass")
    if has_gold:
        OVR += f",retrained_model_path={IU_RE3_DIR}"
    if extra_ovr:
        OVR += "," + extra_ovr
    HYP = f"LoKU IU FORGET {forget_pct}% Kaggle (multiclass head), multi-seed mean+-std."

    for i, s in enumerate(seeds):
        if not force_redo and _iu_seed_done(forget_pct, s):
            print(f"⏭️  SEED {s} đã có trong CSV — skip\n")
            continue
        print(f"\n{'='*60}\n🎲 SEED {s} ({i+1}/{len(seeds)}) ▸ LoKU IU {forget_pct}%\n{'='*60}")
        cmd = (f'PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py '
               f'--config config_loku_iu_kaggle.yaml --fresh --seed {s} '
               f'--override "{OVR}" --exp {exp_name}_seed{s} --hypothesis "{HYP}"')
        get_ipython().system(cmd)
    _iu_aggregate(forget_pct, seeds, exp_name, has_gold)


def _iu_aggregate(forget_pct, seeds, exp_name, has_gold):
    if not os.path.exists(CSV_PATH_IU):
        print(f"❌ {CSV_PATH_IU} chưa có")
        return
    df = pd.read_csv(CSV_PATH_IU)
    if 'forget_pct' in df.columns:
        df = df[df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per")]
    if 'seed' in df.columns:
        df = df[df['seed'].isin(seeds)]
    if df.empty:
        print("❌ Chưa có rows IU cho cấu hình này")
        return
    if 'timestamp' in df.columns:
        df = df.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')
    print(f"\n{'='*90}\n📊 LoKU IU — FORGET {forget_pct}% — seeds={list(seeds)}\n{'='*90}")
    md = [f"# LoKU IU Kaggle — FORGET {forget_pct}%", "",
          f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}_", "",
          "**Dataset**: Indiana University CXR (binary normal/abnormal, head multiclass 4-class)", "",
          f"**Seeds**: {list(seeds)}  |  **Gold retrain**: {'✅' if has_gold else '❌ N/A'}", "",
          "| Metric | " + " | ".join(f"seed {s}" for s in seeds) + " | **mean ± std** |",
          "|---|" + "---|" * (len(seeds) + 1)]
    for csv_key, label, _, _ in METRIC_DEFS:
        if csv_key not in df.columns:
            continue
        vals = []
        for s in seeds:
            sub = df[df['seed'] == s]
            if sub.empty:
                continue
            try:
                v = float(sub[csv_key].iloc[-1])
            except Exception:
                continue
            if v == v:
                vals.append(v)
        if not vals:
            continue
        m, sd = float(np.mean(vals)), float(np.std(vals))
        inv = " ⚠️" if (csv_key == 'dist_vs_re' and not has_gold) else ""
        print(f"{label+inv:<20}" + "".join(f"{v:>10.3f}" for v in vals) + f"{m:>10.3f}±{sd:.3f}")
        md.append(f"| {label}{inv} | " + " | ".join(f"{v:.3f}" for v in vals) + f" | **{m:.3f} ± {sd:.3f}** |")
    os.makedirs("experiments", exist_ok=True)
    out_md = f"experiments/summary_{exp_name}_kaggle_multiseed.md"
    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join(md))
    shutil.copy(CSV_PATH_IU, "experiments/results_summary_loku_iu_kaggle.csv")
    print(f"\n💾 Summary: {out_md}")
    print("💾 CSV → experiments/results_summary_loku_iu_kaggle.csv (Cell 6 sẽ push)")


# ⚙️ Chạy LoKU IU 3% multi-seed (Set RUN_IU_LOKU=False để skip)
RUN_IU_LOKU = False
SEEDS_IU = (42, 123, 7)
if RUN_IU_LOKU:
    run_loku_iu(forget_pct=3, seeds=SEEDS_IU, ihl=1.25)
else:
    print("⏭️  Skip LoKU IU (RUN_IU_LOKU=False)")

In [ ]:
# ====================================
# CELL 4d-SWEEP (IU): thu nhieu cach QUEN cho LoKU IU 3% -> chon tot nhat
# ====================================
# Config D honest (IHL-only) KHONG quen duoc tren IU (IHL bao hoa). Sweep cac co che:
#  A/B/E: distill-toward-RE (chia khoa) | C: IHL cao | D: NegGrad (kappa_cls_forget)
RUN_IU_SWEEP = True
IU_SWEEP = [
    {'tag': 'A_distillre',    'ihl': 1.25, 'dt': 're', 'df': 1.0, 'kf': 0.0, 'es': 'cossim', 'ep': 15},
    {'tag': 'B_distillre_hi', 'ihl': 1.25, 'dt': 're', 'df': 2.0, 'kf': 0.0, 'es': 'cossim', 'ep': 15},
    {'tag': 'C_ihl_high',     'ihl': 3.0,  'dt': 'og', 'df': 0.0, 'kf': 0.0, 'es': 'none',   'ep': 15},
    {'tag': 'D_neggrad',      'ihl': 1.25, 'dt': 'og', 'df': 0.0, 'kf': 0.5, 'es': 'none',   'ep': 15},
    {'tag': 'E_combo',        'ihl': 2.0,  'dt': 're', 'df': 1.0, 'kf': 0.3, 'es': 'cossim', 'ep': 20},
]
if RUN_IU_SWEEP:
    for _c in IU_SWEEP:
        _ovr = ('distill_teacher=' + _c['dt'] + ',distill_forget_weight=' + str(_c['df']) +
                ',kappa_cls_forget=' + str(_c['kf']) + ',early_stop_metric=' + _c['es'] +
                ',unlearn_epochs=' + str(_c['ep']))
        run_loku_iu(forget_pct=3, seeds=(42,), ihl=_c['ihl'],
                    tag='_' + _c['tag'], extra_ovr=_ovr, force_redo=True)
else:
    print('skip IU sweep')

In [ ]:
# ====================================
# CELL 5: Bảng cross-forget% — LoKU Kaggle
# ====================================
import os, pandas as pd
from datetime import datetime

if not os.path.exists(CSV_PATH):
    print(f"❌ {CSV_PATH} không tồn tại. Cần chạy ít nhất 1 Cell 4*.")
else:
    df_all = pd.read_csv(CSV_PATH)
    pcts_in_csv = sorted({int(s.split('_')[-1].replace('per.csv', '').replace('per', ''))
                          for s in df_all['forget_pct'].dropna().astype(str).unique() if '_' in s})
    print(f"📋 Total rows: {len(df_all)} — forget% có data: {pcts_in_csv}\n")

    show_metrics = [
        ('MIA_paper', 'MIA', '↓'), ('Df_AUC', 'Df_AUC', '↓'), ('Df_F1', 'Df_F1', '↓'),
        ('Dt_AUC', 'Dt_AUC', '↑'), ('Dt_F1', 'Dt_F1', '↑'),
        ('dist_vs_re', '1−CosSim', '↓'), ('unlearn_time_hours', 'Time(h)', '↓'),
    ]
    md_lines = [
        "# LoKU Kaggle — Bảng cross-forget%",
        "",
        f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}_",
        "",
        "Kết quả LoKU multi-seed trên Kaggle T4 (so trực tiếp với baseline cùng GPU).",
        "",
    ]
    header = "| Forget% | Method | " + " | ".join(m[1] for m in show_metrics) + " |"
    sep = "|---|---|" + "---|" * len(show_metrics)
    md_lines += [header, sep]
    print("=" * 110); print(header); print("=" * 110)

    paper_key_map = {'MIA_paper': 'MIA_paper', 'Df_AUC': 'Df_AUC', 'Df_F1': 'Df_F1',
                     'Dt_AUC': 'Dt_AUC', 'Dt_F1': 'Dt_F1', 'unlearn_time_hours': 'Time_h'}

    for pct in [3, 6, 10]:
        sub = df_all[df_all['forget_pct'].astype(str).str.contains(f"_{pct}per")]
        if sub.empty:
            row = f"| {pct}% | _(chưa chạy)_ |" + " — |" * len(show_metrics)
            md_lines.append(row); print(row); continue

        if 'timestamp' in sub.columns:
            sub = sub.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')
        n = sub['seed'].nunique()

        # Paper row
        paper = PAPER_REF.get(pct, {})
        paper_row = f"| {pct}% | Paper Forget-MI |"
        for csv_k, _, _ in show_metrics:
            pk = paper_key_map.get(csv_k)
            paper_row += f" {paper.get(pk, '—')} |" if pk and pk in paper else " — |"
        md_lines.append(paper_row); print(paper_row)

        # LoKU row (mean of seeds)
        loku_row = f"| {pct}% | LoKU (n={n}) |"
        for csv_k, _, _ in show_metrics:
            if csv_k in sub.columns:
                v = sub[csv_k].mean()
                loku_row += f" {v:.3f} |" if pd.notna(v) else " — |"
            else:
                loku_row += " — |"
        md_lines.append(loku_row); print(loku_row)
        md_lines.append("")

    out_md = "experiments/bang_loku_kaggle_cross_forget.md"
    os.makedirs("experiments", exist_ok=True)
    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join(md_lines))
    print(f"\n💾 Saved: {out_md}")


In [ ]:
# ====================================
# CELL 6: Push LoKU results lên GitHub (Kaggle Secrets)
# ====================================
import os, shutil

GITHUB_REPO = "nhnhu146/Forget-MI-LoKU"
BRANCH = "master"

# BƯỚC 1: Copy CSV
CSV_SRC = "/kaggle/working/unlearning_output/results_summary.csv"
CSV_DST = "experiments/results_summary_loku_kaggle.csv"
if os.path.exists(CSV_SRC):
    os.makedirs("experiments", exist_ok=True)
    shutil.copy(CSV_SRC, CSV_DST)
    n = sum(1 for _ in open(CSV_DST)) - 1
    print(f"📋 Copied CSV ({n} rows) → {CSV_DST}")
else:
    print(f"ℹ️  CSV chưa có ({CSV_SRC}) — chỉ push MD files")

# BƯỚC 2: Load credentials từ Kaggle Secrets
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        return (secrets.get_secret('GITHUB_TOKEN'),
                secrets.get_secret('GIT_EMAIL'),
                secrets.get_secret('GIT_NAME'))
    except Exception as e:
        print(f"⚠️  Không load được Kaggle Secrets ({e})")
        return None, None, None

TOKEN, EMAIL, NAME = load_secrets()
if TOKEN and EMAIL and NAME:
    print("🔑 Credentials từ Kaggle Secrets ✅")
else:
    print("⚠️  Thiếu credentials — skip push")
    raise SystemExit

# BƯỚC 3: Git commit + push
get_ipython().system(f'git config user.email "{EMAIL}"')
get_ipython().system(f'git config user.name "{NAME}"')
get_ipython().system(f'git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git')
get_ipython().system('git add experiments/ 2>/dev/null')

changes = get_ipython().getoutput('git diff --cached --name-only')
if changes and any(c.strip() for c in changes):
    print("\n📦 Files commit:")
    for f in changes:
        if f.strip():
            print(f"   - {f}")
    msg = "loku kaggle: multi-seed results + CSV checkpoint"
    get_ipython().system(f'git commit -m "{msg}"')
    get_ipython().system(f'git push origin {BRANCH}')
    print(f"\n✅ Pushed: https://github.com/{GITHUB_REPO}/tree/{BRANCH}/experiments")
else:
    print("ℹ️  Không có file mới để commit.")
